# ETAP AI Platform — Arc Flash Analysis Demo

This notebook demonstrates how to perform an **Arc Flash Hazard Analysis** compliant with **IEEE 1584-2018** and **NFPA 70E** using the ETAP AI Engineering Platform API. Arc flash analysis is critical for worker safety, determining the incident energy, arc flash boundary, and required personal protective equipment (PPE) at each equipment location.

## What You'll Learn
- How to configure arc flash analysis parameters
- How to interpret incident energy and PPE category results
- How to compare results for different electrode configurations
- How to apply the IEEE 1584-2018 and NFPA 70E safety requirements

## Standards Covered
- IEEE 1584-2018 — Guide for Performing Arc-Flash Hazard Calculations
- NFPA 70E-2024 — Standard for Electrical Safety in the Workplace
- IEEE 1584.1-2022 — Guide for the Specification of Scope and Deliverables

In [ ]:
# Cell 1: Setup

import matplotlib.pyplot as plt
import requests

BASE_URL = "http://localhost:8000/api/v1"
API_KEY = "your-api-key-here"
HEADERS = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

print("Arc Flash Analysis Demo — IEEE 1584-2018 & NFPA 70E")

## Step 1: Define the System and Fault Current

Arc flash analysis requires knowing the available bolted fault current at each equipment location. We first define the power system and then specify the arc flash parameters for the equipment of interest.

In [ ]:
# Cell 2: Define power system for fault current calculation
system_data = {
    "base_mva": 100.0,
    "buses": [
        {"bus_id": 1, "bus_type": "slack", "voltage_magnitude": 1.05, "base_kv": 13.8},
        {"bus_id": 2, "bus_type": "pq", "base_kv": 4.16},
        {"bus_id": 3, "bus_type": "pq", "base_kv": 0.48},
    ],
    "lines": [
        {"line_id": 1, "from_bus_id": 1, "to_bus_id": 2, "r1": 0.01, "x1": 0.05, "bshunt1": 0.02},
        {"line_id": 2, "from_bus_id": 2, "to_bus_id": 3, "r1": 0.005, "x1": 0.04, "bshunt1": 0.01},
    ],
    "generators": [{"generator_id": 1, "bus_id": 1, "x1": 0.15, "internal_voltage_mag": 1.05}],
}

# Arc flash parameters for equipment at Bus 2 (4.16 kV Switchgear)
arc_flash_params = {
    "voltage_kv": 4.16,
    "bolted_fault_current_ka": 25.0,
    "arc_duration_sec": 0.5,
    "working_distance_mm": 610.0,
    "equipment_type": "switchgear",
    "electrode_configuration": "VCB",
    "grounding_type": "solidly_grounded",
}

print("Arc Flash Parameters:")
print(f"  Voltage:          {arc_flash_params['voltage_kv']} kV")
print(f"  Bolted Fault:     {arc_flash_params['bolted_fault_current_ka']} kA")
print(f"  Arc Duration:     {arc_flash_params['arc_duration_sec']} s")
print(f"  Working Distance: {arc_flash_params['working_distance_mm']} mm")
print(f"  Electrode Config: {arc_flash_params['electrode_configuration']}")

## Step 2: Run Arc Flash Analysis

Submit the arc flash study to the API. The IEEE 1584-2018 empirical model calculates incident energy based on voltage, arcing current, gap distance, electrode configuration, and enclosure dimensions.

In [ ]:
# Cell 3: Run arc flash analysis
study_request = {"study_type": "arc_flash", "system": system_data, "parameters": arc_flash_params}

try:
    response = requests.post(
        f"{BASE_URL}/studies/run", json=study_request, headers=HEADERS, timeout=60,
    )
    result = response.json()
    af_results = result.get("results", {})
except requests.exceptions.ConnectionError:
    # Sample results for demonstration
    af_results = {
        "arcing_current_ka": 20.5,
        "arc_current_variation_85_percent_ka": 17.4,
        "incident_energy_cal_cm2": 12.3,
        "arc_flash_boundary_mm": 2100,
        "ppe_level": "Category 3",
        "minimum_ppe_rating_cal_cm2": 25.0,
        "limited_approach_boundary_ft": 3.5,
        "restricted_approach_boundary_ft": 1.0,
        "electrode_configuration": "VCB",
        "recommendations": [
            "Use Category 3 PPE minimum (25 cal/cm²)",
            "Maintain safe working distance of 610 mm",
            "Arc flash boundary: 2.1 m — barricade required",
        ],
    }
    print("(Using sample results for demonstration)")

# Display results
print("\n" + "=" * 55)
print("  Arc Flash Analysis Results (IEEE 1584-2018)")
print("=" * 55)
print(f"  Arcing Current:              {af_results.get('arcing_current_ka', 0):.1f} kA")
print(
    f"  85% Arc Current:             {af_results.get('arc_current_variation_85_percent_ka', 0):.1f} kA",
)
print(f"  Incident Energy:             {af_results.get('incident_energy_cal_cm2', 0):.1f} cal/cm²")
print(
    f"  Arc Flash Boundary:          {af_results.get('arc_flash_boundary_mm', 0):.0f} mm ({af_results.get('arc_flash_boundary_mm', 0) / 1000:.1f} m)",
)
print(f"  PPE Level:                   {af_results.get('ppe_level', 'N/A')}")
print(
    f"  Minimum PPE Rating:          {af_results.get('minimum_ppe_rating_cal_cm2', 0):.1f} cal/cm²",
)
print(f"  Limited Approach Boundary:   {af_results.get('limited_approach_boundary_ft', 0)} ft")
print(f"  Restricted Approach Boundary:{af_results.get('restricted_approach_boundary_ft', 0)} ft")
print("=" * 55)

## Step 3: Compare Electrode Configurations

IEEE 1584-2018 defines five electrode configurations, each producing different incident energy levels. It's important to select the correct configuration for the equipment being analyzed. Let's compare all five configurations for the same fault current.

In [ ]:
# Cell 4: Compare electrode configurations
electrode_configs = ["VCB", "VCBB", "HCB", "VOA", "HOA"]
config_results = {}

# Sample results for all configurations (replace with API calls in production)
sample_config_results = {
    "VCB": {"incident_energy": 12.3, "boundary_mm": 2100},
    "VCBB": {"incident_energy": 15.8, "boundary_mm": 2700},
    "HCB": {"incident_energy": 18.2, "boundary_mm": 3200},
    "VOA": {"incident_energy": 8.5, "boundary_mm": 1500},
    "HOA": {"incident_energy": 10.1, "boundary_mm": 1800},
}

for config in electrode_configs:
    try:
        params = arc_flash_params.copy()
        params["electrode_configuration"] = config
        study_request["parameters"] = params
        response = requests.post(
            f"{BASE_URL}/studies/run", json=study_request, headers=HEADERS, timeout=60,
        )
        result = response.json()
        res = result.get("results", {})
        config_results[config] = {
            "incident_energy": res.get("incident_energy_cal_cm2", 0),
            "boundary_mm": res.get("arc_flash_boundary_mm", 0),
        }
    except requests.exceptions.ConnectionError:
        config_results = sample_config_results
        print("(Using sample results for demonstration)")
        break

# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Incident Energy
energies = [config_results[c]["incident_energy"] for c in electrode_configs]
colors = [
    "green" if e <= 4 else ("orange" if e <= 8 else ("coral" if e <= 25 else "red"))
    for e in energies
]
ax1.bar(electrode_configs, energies, color=colors, edgecolor="black", alpha=0.8)
ax1.set_xlabel("Electrode Configuration")
ax1.set_ylabel("Incident Energy (cal/cm²)")
ax1.set_title("Incident Energy by Electrode Configuration")
ax1.grid(True, alpha=0.3, axis="y")

# Add PPE zone lines
ax1.axhline(y=4, color="green", linestyle="--", alpha=0.5, label="Cat 1 (4 cal/cm²)")
ax1.axhline(y=8, color="orange", linestyle="--", alpha=0.5, label="Cat 2 (8 cal/cm²)")
ax1.axhline(y=25, color="red", linestyle="--", alpha=0.5, label="Cat 3 (25 cal/cm²)")
ax1.legend(fontsize=8)

# Arc Flash Boundary
boundaries = [config_results[c]["boundary_mm"] / 1000 for c in electrode_configs]
ax2.bar(electrode_configs, boundaries, color="steelblue", edgecolor="black", alpha=0.8)
ax2.set_xlabel("Electrode Configuration")
ax2.set_ylabel("Arc Flash Boundary (m)")
ax2.set_title("Arc Flash Boundary by Electrode Configuration")
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("arc_flash_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 4: NFPA 70E PPE Categories

NFPA 70E defines PPE categories based on incident energy levels. The correct PPE must be worn when working within the arc flash boundary of energized equipment.

In [ ]:
# Cell 5: PPE Category Reference and Recommendation
ppe_categories = {
    "Category 1": {
        "min_cal": 4.0,
        "max_cal": 7.9,
        "description": "Arc-rated shirt and pants, safety glasses, hearing protection, voltage-rated gloves",
    },
    "Category 2": {
        "min_cal": 8.0,
        "max_cal": 24.9,
        "description": "Arc-rated shirt and pants, arc-rated face shield, hearing protection, voltage-rated gloves",
    },
    "Category 3": {
        "min_cal": 25.0,
        "max_cal": 39.9,
        "description": "Arc-rated flash suit hood, arc-rated shirt and pants, hearing protection, voltage-rated gloves",
    },
    "Category 4": {
        "min_cal": 40.0,
        "max_cal": 104.0,
        "description": "Arc-rated flash suit hood, arc-rated multi-layer flash suit, hearing protection, voltage-rated gloves",
    },
}

incident_energy = af_results.get("incident_energy_cal_cm2", 0)

print("NFPA 70E PPE Category Reference:")
print("=" * 70)
for cat, info in ppe_categories.items():
    marker = " ◀ YOUR LEVEL" if info["min_cal"] <= incident_energy <= info["max_cal"] else ""
    print(f"\n{cat} ({info['min_cal']:.0f}–{info['max_cal']:.0f} cal/cm²){marker}")
    print(f"   {info['description']}")

print(f"\n{'=' * 70}")
print(f"Your incident energy: {incident_energy:.1f} cal/cm²")
print(f"Recommended PPE: {af_results.get('ppe_level', 'N/A')}")
print(f"Minimum rating: {af_results.get('minimum_ppe_rating_cal_cm2', 0):.1f} cal/cm²")

## Step 5: Safety Recommendations

Based on the arc flash analysis results, the following safety measures should be implemented per NFPA 70E and IEEE 1584-2018.

In [ ]:
# Cell 6: Safety Recommendations
recommendations = af_results.get("recommendations", [])

print("🔒 Safety Recommendations (NFPA 70E & IEEE 1584-2018)")
print("=" * 55)
for i, rec in enumerate(recommendations, 1):
    print(f"  {i}. {rec}")

print("\n📋 Additional Requirements:")
print("  • Label all equipment with arc flash warning per NFPA 70E 130.5(H)")
print("  • Include incident energy, boundary, PPE level, and voltage on labels")
print("  • Perform job briefing before energized work per NFPA 70E 130.2")
print("  • Establish arc flash boundary and barricade per NFPA 70E 130.5")
print("  • Document analysis date and revision per IEEE 1584.1-2022")
print("  • Re-evaluate when system changes occur (max 5-year review per NFPA 70E)")

## Summary

In this demo, you learned how to:
1. Configure arc flash analysis parameters per IEEE 1584-2018
2. Calculate incident energy and arc flash boundaries
3. Compare results across different electrode configurations
4. Determine the correct PPE category per NFPA 70E
5. Apply safety recommendations for worker protection

**Important:** Arc flash analysis must be performed by qualified personnel and reviewed periodically. Always verify results against field measurements and consult the full IEEE 1584-2018 and NFPA 70E standards for complete requirements.